<a href="https://colab.research.google.com/github/ruben-tsui/Translation_Tech/blob/main/sentBertAlignTMXQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Bertalign**: Bitext Alignment at the ***Sentence*** Level

### Expand for More Information


`https://github.com/bfsujason/bertalign`

Bertalign is a sentence alignment module for creating parallel corpora and translation memories. It uses multilingual sentence transformer models to align bilingual sentences and documents. Refer to the following paper for details:

`https://academic.oup.com/dsh/article-abstract/38/2/621/6965034`

I've forked the above and made certain modifications to improve sentence tokenization for Chinese. It's available from:

`https://github.com/ruben-tsui/bertalign`

#### This version last updated on 2026-04-08
* Added TMX output support

## Step (1): Update system libraries (requries a restart; *ignore the ***system crash*** message*)

In [ ]:
%%capture
!pip install faiss-gpu-cu12 sentence-splitter==1.4
exit()

## Step (2): Retrieve alignment libraries and embedding model

In [ ]:
# "Clone" the source code for the BertAlign aligner
!git clone https://github.com/TranslationTech/bertalignQ

In [ ]:
cd /content/bertalignQ

In [ ]:
from bertalign import Bertalign

In [ ]:
cd ..

## Step (3): Upload your source and target texts. Enter input file names, source and target languages, `max_align` parameter below

##### For subsequent text alignments within the same Colab session, modify the input file names and other parameters and do a `Runtime -> Run cell and below` from Step (3).

In [ ]:
#### USER INPUTS ####
# (1) Your source and target input file names must conform to the format:
#     sample.en.txt
#     sample.zh.txt
#     and you will enter the base file name as follows
base = '/content/sample'
base = 'sample' # @param {"type":"string"}

# (2) Considers all "N sentences-to-M sentences" alignment, N + M <= max_align;
#     The larger this value, the longer it will take to complete the alignment.
#max_align = 7 #  @param [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
max_align = 7 # @param {type:"slider", min:2, max:20, step:1}

# (3) Source and target languages (check ISO 689-1 for language codes)
# source language, e.g. en, zh, es, fr, de, ja, it
src_lang = 'zh'   # @param ['en', 'zh', 'fr', 'es', 'ja', 'it']
# target language
tgt_lang = 'en'   # @param ['en', 'zh', 'fr', 'es', 'ja', 'it']

#### DO NOT CHANGE THE FOLLOWING ####
# input files (source and target texts)
fin_src  = f'{base}.{src_lang}.txt'
fin_tgt  = f'{base}.{tgt_lang}.txt'
# output (aligned) file (plain text and Excel)
fon_txt  = f'{base}.bertalign.n{max_align}.{src_lang}-{tgt_lang}.txt'
fon_xlsx = f'{base}.bertalign.n{max_align}.{src_lang}-{tgt_lang}.xlsx'


## Step (4) Run the Alignment!

In [ ]:
print(f"Reading source text: {fin_src}...")
src = open(fin_src, 'r', encoding='utf-8').read()
print(f"Reading target text: {fin_tgt}...")
tgt = open(fin_tgt, 'r', encoding='utf-8').read()

### Sentence aligning begins below

In [ ]:
%%time
aligner = Bertalign(src, tgt, max_align=max_align, src_lang=src_lang, tgt_lang=tgt_lang)
aligner.align_sents()

In [ ]:
from bertalign import model
from bertalign.utils import *
import numpy as np

### Creates the sentence-aligned text in .txt (plain text) format

In [ ]:
data = []
with open(fon_txt, 'w', encoding='utf-8', newline='\n') as fo:
    cnt = 0
    for s, t in aligner.result:
        cnt += 1
        s = [int(x) for x in s]
        t = [int(x) for x in t]
        # source
        if src_lang in ['zh', 'ja', 'kr']:
            s_delim = ''  # no space between sentences
        else:
            s_delim = ' ' # one space between sentences
        ss = [aligner.src_sents[sidx].strip() for sidx in s]
        ss = s_delim.join(ss)
        sv = model.model.encode(ss, normalize_embeddings=True)
        # target
        if tgt_lang in ['zh', 'ja', 'kr']:
            t_delim = ''  # no space between sentences
        else:
            t_delim = ' ' # one space between sentences
        tt = [aligner.tgt_sents[tidx].strip() for tidx in t]
        tt = t_delim.join(tt)
        tv = model.model.encode(tt, normalize_embeddings=True)
        #
        cossim = np.dot(sv, tv)
        cosdist = 1 - cossim
        line = f"{cosdist:.4f}\t{str(s)}\t{ss}\t{str(t)}\t{tt}\n"
        data.append(line)
        fo.write(line)

Creates the sentence-aligned text in .xlsx (Microsoft Excel) format

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Border, Side, Alignment, Protection, Font
from openpyxl.utils.dataframe import dataframe_to_rows

# Create a new workbook
wb = openpyxl.Workbook()
# Select the active sheet
ws = wb.active
# Set column widths
ws.column_dimensions['A'].width = 10
ws.column_dimensions['B'].width = 10
ws.column_dimensions['C'].width = 10
ws.column_dimensions['D'].width = 50
ws.column_dimensions['E'].width = 10
ws.column_dimensions['F'].width = 65

df = pd.DataFrame([x.split('\t') for x in data], columns=['cosdist', 'cols_s', src_lang, 'cols_t',  tgt_lang])

for r in dataframe_to_rows(df, index=True, header=True):
    ws.append(r)

# Set cell alignment
alignment = Alignment(horizontal='general',
                      vertical='top',
                      wrap_text=True)

for row in ws[f'A1:F{cnt+10}']:
    for cell in row:
        cell.alignment = alignment

# Save the workbook
wb.save(fon_xlsx)


Creates the sentence-aligned text in .tmx (TMX - Translation Memory eXchange) format

In [ ]:
import csv
import sys
import xml.etree.ElementTree as ET
from xml.dom import minidom

def create_tmx(input_file, output_file, src_lang, tgt_lang):
    """Create a TMX file from a tab-delimited alignment file."""

    if src_lang == 'zh':
        src_lang = 'zh-TW'
    if tgt_lang == 'zh':
        tgt_lang = 'zh-TW'

    tmx = ET.Element("tmx", version="1.4")

    header = ET.SubElement(
        tmx,
        "header",
        {
            "creationtool": "TMX_E2C",
            "creationtoolversion": "1.0",
            "segtype": "sentence",
            "o-tmf": "unknown",
            "adminlang": "en",
            "srclang": src_lang,
            "datatype": "PlainText",
        },
    )

    body = ET.SubElement(tmx, "body")

    with open(input_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) < 5:
                continue

            score = row[0].strip()
            en_text = row[2].strip()
            zh_text = row[4].strip()

            tu = ET.SubElement(body, "tu")

            prop = ET.SubElement(tu, "prop", {"type": "cosine-distance"})
            prop.text = score

            tuv_en = ET.SubElement(tu, "tuv", {"xml:lang": src_lang})
            seg_en = ET.SubElement(tuv_en, "seg")
            seg_en.text = en_text

            tuv_zh = ET.SubElement(tu, "tuv", {"xml:lang": tgt_lang})
            seg_zh = ET.SubElement(tuv_zh, "seg")
            seg_zh.text = zh_text

    rough_string = ET.tostring(tmx, encoding="unicode", xml_declaration=False)
    dom = minidom.parseString(rough_string)
    pretty = dom.toprettyxml(indent="  ", encoding="UTF-8")

    with open(output_file, "wb") as f:
        f.write(pretty)

    print(f"TMX file created: {output_file}")

create_tmx(fon_txt, fon_txt.replace('.txt', '.tmx'), src_lang, tgt_lang)